In [18]:
#Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, roc_auc_score

In [19]:
#File path to the zipped datafile
zip_path = '../../Data/Processed/Data_Compressed.zip'

#List of filenames to read from the zip file
filenames = [
    'Data_Compressed/all_normalized_features.csv',
    'Data_Compressed/kdd_all_normalized_features.csv',
    'Data_Compressed/kdd_expanded_all_scaled.csv',
    'Data_Compressed/kdd_merged_normalized_all.csv'
]

#Corresponding names for the DataFrames
df_names = [
    'xuetangx_df',
    'kdd_df',
    'kdd_expanded_df',
    'kdd_merged_df'
]

#Create an empty dictionary to store the DataFrames
dataframes = {}

#Open the zip file
z = zipfile.ZipFile(zip_path, 'r')

#Loop through the filenames and read them into DataFrames
for i, file in enumerate(filenames):
    print(f"Reading file: {file} from {zip_path}")
    
    #Read each CSV file directly from the zip
    f = z.open(file)
    dataframes[df_names[i]] = pd.read_csv(f)
    f.close()
    
    print(f"{df_names[i]} loaded with shape: {dataframes[df_names[i]].shape}\n")

#Close the zip file after reading
z.close()

#Assign individual DataFrames to variables
xuetangx_df = dataframes['xuetangx_df']
kdd_df = dataframes['kdd_df']
kdd_expanded_df = dataframes['kdd_expanded_df']
kdd_merged_df = dataframes['kdd_merged_df']


Reading file: Data_Compressed/all_normalized_features.csv from ../../Data/Processed/Data_Compressed.zip
xuetangx_df loaded with shape: (225642, 30)

Reading file: Data_Compressed/kdd_all_normalized_features.csv from ../../Data/Processed/Data_Compressed.zip
kdd_df loaded with shape: (200904, 17)

Reading file: Data_Compressed/kdd_expanded_all_scaled.csv from ../../Data/Processed/Data_Compressed.zip
kdd_expanded_df loaded with shape: (120542, 142)

Reading file: Data_Compressed/kdd_merged_normalized_all.csv from ../../Data/Processed/Data_Compressed.zip
kdd_merged_df loaded with shape: (120542, 158)



In [20]:
# %pip install tensorflow
# %pip install keras
# %pip install scikit-learn
# %pip install fastai
# %pip install torch

In [21]:
#Import required libraries for Deep Learning
#Keras DNN Classifier
from keras.models import Sequential
from keras.layers import BatchNormalization, Dense, Dropout, Input
from keras.regularizers import l2
from keras.utils import to_categorical, normalize
from keras import backend as K

#FastAI DL Classifier
import torch
from fastai.tabular.all import *

#Metrics for evaluation
from sklearn.metrics import balanced_accuracy_score, accuracy_score, precision_score, recall_score, roc_auc_score, f1_score
from sklearn.model_selection import train_test_split

### Data Preprocessing

In [22]:
#Display the info of the xuetangx_df DataFrame
xuetangx_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 225642 entries, 0 to 225641
Data columns (total 30 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   enroll_id                      225642 non-null  int64  
 1   action_count                   225642 non-null  float64
 2   seek_video_count               225642 non-null  float64
 3   play_video_count               225642 non-null  float64
 4   pause_video_count              225642 non-null  float64
 5   stop_video_count               225642 non-null  float64
 6   load_video_count               225642 non-null  float64
 7   problem_get_count              225642 non-null  float64
 8   problem_check_count            225642 non-null  float64
 9   problem_save_count             225642 non-null  float64
 10  reset_problem_count            225642 non-null  float64
 11  problem_check_correct_count    225642 non-null  float64
 12  problem_check_incorrect_count 

In [23]:
#Display the info of the kdd_df DataFrame
kdd_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200904 entries, 0 to 200903
Data columns (total 17 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   enrollment_id             200904 non-null  int64  
 1   action_count              200904 non-null  float64
 2   server_navigate_count     200904 non-null  float64
 3   server_access_count       200904 non-null  float64
 4   server_problem_count      200904 non-null  float64
 5   server_page_close_count   200904 non-null  float64
 6   server_video_count        200904 non-null  float64
 7   server_discussion_count   200904 non-null  float64
 8   server_wiki_count         200904 non-null  float64
 9   browser_navigate_count    200904 non-null  float64
 10  browser_access_count      200904 non-null  float64
 11  browser_problem_count     200904 non-null  float64
 12  browser_page_close_count  200904 non-null  float64
 13  browser_video_count       200904 non-null  f

In [24]:
#Display the info of the kdd_expanded_df DataFrame
kdd_expanded_df.columns

Index(['Unnamed: 0', 'enrollment_id', 'truth', 'avg_chapter_delays',
       'server_discussion_percent', 'act_cnt_weekDay_01',
       'browser_html_percent', 'parallel_enrollments', 'browser_dictation',
       'act_cnt_day_00',
       ...
       'server_course_percent', 'browser_course_info_percent',
       'browser_course', 'browser_vertical_percent', 'sessions_in_week_1',
       'sessions_in_week_0', 'sessions_in_week_3', 'sessions_in_week_2',
       'sessions_in_week_4', 'browser_about'],
      dtype='object', length=142)

In [25]:
#Isolate the X and y features for the xuetangx dataset
xuetangx_X = xuetangx_df.drop(columns=['truth'])
xuetangx_X = xuetangx_X.drop(columns=['enroll_id'])
xuetangx_y = xuetangx_df['truth']
#Isolate the X and y features for the kdd dataset
kdd_X = kdd_df.drop(columns=['truth'])
kdd_X = kdd_X.drop(columns=['enrollment_id'])
kdd_y = kdd_df['truth']
#Isolate the X and y features for the kdd_expanded 
kdd_expanded_X = kdd_expanded_df.drop(columns=['truth'])
kdd_expanded_X = kdd_expanded_X.drop(columns=['enrollment_id'])
kdd_expanded_X = kdd_expanded_X.drop(columns=['Unnamed: 0'])
kdd_expanded_y = kdd_expanded_df['truth']

In [26]:
#Split the xuetangx dataset into training and testing sets
xuetangx_X_train, xuetangx_X_test, xuetangx_y_train, xuetangx_y_test = train_test_split(xuetangx_X, xuetangx_y, test_size=0.2, random_state=100)

#Split the kdd dataset into training and testing sets
kdd_X_train, kdd_X_test, kdd_y_train, kdd_y_test = train_test_split(kdd_X, kdd_y, test_size=0.2, random_state=100)

#Split the kdd_expanded dataset into training and testing sets
kdd_expanded_X_train, kdd_expanded_X_test, kdd_expanded_y_train, kdd_expanded_y_test = train_test_split(kdd_expanded_X, kdd_expanded_y, test_size=0.2, random_state=100)

### Keras-TensorFlow

#### Xuetangx

In [27]:
#Modify the training and test variables to use the xuetangx dataset for readability
X_train, X_test, y_train, y_test = xuetangx_X_train, xuetangx_X_test, xuetangx_y_train, xuetangx_y_test

In [28]:
#Initialize KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

#Prepare results storage
results = {
    'Accuracy': [],
    'Balanced Accuracy': [],
    'Recall': [],
    'Precision': [],
    'AUC': [],
    'F1 Score': []
}

X_np = X_train.values
y_np = y_train.values

for train_index, val_index in kf.split(X_np):
    X_train_fold, X_val_fold = X_np[train_index], X_np[val_index]
    y_train_fold, y_val_fold = y_np[train_index], y_np[val_index]

    #Build and compile model
    model = Sequential([
        Dense(128, kernel_regularizer=l2(0.001), activation='relu', input_shape=(X_train.shape[1],)),
        BatchNormalization(),
        Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dense(len(np.unique(y_train)), activation='softmax')
    ])
    
    model.compile(optimizer='adam', loss='categorical_crossentropy')

    #Convert y to one-hot for training
    y_train_cat = to_categorical(y_train_fold)
    
    #Fit the model
    model.fit(X_train_fold, y_train_cat, epochs=100, verbose=0, batch_size=512)

    #Predict and evaluate
    y_val_probs = model.predict(X_val_fold)
    y_pred = np.argmax(y_val_probs, axis=1)

    #Metrics
    acc = accuracy_score(y_val_fold, y_pred) * 100
    bal_acc = balanced_accuracy_score(y_val_fold, y_pred) * 100
    rec = recall_score(y_val_fold, y_pred, average='weighted') * 100
    prec = precision_score(y_val_fold, y_pred, average='weighted') * 100
    auc = roc_auc_score(y_val_fold, y_pred) * 100
    f1 = f1_score(y_val_fold, y_pred) * 100

    #Store metrics
    results['Accuracy'].append(acc)
    results['Balanced Accuracy'].append(bal_acc)
    results['Recall'].append(rec)
    results['Precision'].append(prec)
    results['AUC'].append(auc)
    results['F1 Score'].append(f1)

#Summarize results
summary = {
    metric: f"{np.mean(vals):.2f} ± {np.std(vals):.2f}"
    for metric, vals in results.items()
}



c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1129/1129 ━━━━━━━━━━━━━━━━━━━━ 1s 505us/step


c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1129/1129 ━━━━━━━━━━━━━━━━━━━━ 1s 492us/step


c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1129/1129 ━━━━━━━━━━━━━━━━━━━━ 1s 513us/step


c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1129/1129 ━━━━━━━━━━━━━━━━━━━━ 1s 498us/step


c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1129/1129 ━━━━━━━━━━━━━━━━━━━━ 1s 496us/step


In [29]:
#Store the results in the comparison DataFrame
xuetangx_tf_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
xuetangx_final_df = xuetangx_tf_df.set_index('Metric').T
xuetangx_final_df.index = ['Keras-Tensorflow'] 
print(xuetangx_final_df)

Metric                Accuracy Balanced Accuracy        Recall     Precision  \
Keras-Tensorflow  83.71 ± 0.10      71.84 ± 0.21  83.71 ± 0.10  82.86 ± 0.14   

Metric                     AUC      F1 Score  
Keras-Tensorflow  71.84 ± 0.21  89.82 ± 0.09  


#### KDD (Experiment 1)

In [30]:
#Modify the training and test variables to use the xuetangx dataset for readability
X_train, X_test, y_train, y_test = kdd_X_train, kdd_X_test, kdd_y_train, kdd_y_test

In [31]:
#Initialize KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

#Prepare results storage
results = {
    'Accuracy': [],
    'Balanced Accuracy': [],
    'Recall': [],
    'Precision': [],
    'AUC': [],
    'F1 Score': []
}

X_np = X_train.values
y_np = y_train.values

for train_index, val_index in kf.split(X_np):
    X_train_fold, X_val_fold = X_np[train_index], X_np[val_index]
    y_train_fold, y_val_fold = y_np[train_index], y_np[val_index]

    #Build and compile model
    model = Sequential([
        Dense(128, kernel_regularizer=l2(0.001), activation='relu', input_shape=(X_train.shape[1],)),
        BatchNormalization(),
        Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dense(len(np.unique(y_train)), activation='softmax')
    ])
    
    model.compile(optimizer='adam', loss='categorical_crossentropy')

    #Convert y to one-hot for training
    y_train_cat = to_categorical(y_train_fold)
    
    #Fit the model
    model.fit(X_train_fold, y_train_cat, epochs=100, verbose=0, batch_size=512)

    #Predict and evaluate
    y_val_probs = model.predict(X_val_fold)
    y_pred = np.argmax(y_val_probs, axis=1)

    #Metrics
    acc = accuracy_score(y_val_fold, y_pred) * 100
    bal_acc = balanced_accuracy_score(y_val_fold, y_pred) * 100
    rec = recall_score(y_val_fold, y_pred, average='weighted') * 100
    prec = precision_score(y_val_fold, y_pred, average='weighted') * 100
    auc = roc_auc_score(y_val_fold, y_pred) * 100
    f1 = f1_score(y_val_fold, y_pred) * 100

    #Store metrics
    results['Accuracy'].append(acc)
    results['Balanced Accuracy'].append(bal_acc)
    results['Recall'].append(rec)
    results['Precision'].append(prec)
    results['AUC'].append(auc)
    results['F1 Score'].append(f1)

#Summarize results
summary = {
    metric: f"{np.mean(vals):.2f} ± {np.std(vals):.2f}"
    for metric, vals in results.items()
}

c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1005/1005 ━━━━━━━━━━━━━━━━━━━━ 1s 479us/step


c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1005/1005 ━━━━━━━━━━━━━━━━━━━━ 1s 484us/step


c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1005/1005 ━━━━━━━━━━━━━━━━━━━━ 1s 504us/step


c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1005/1005 ━━━━━━━━━━━━━━━━━━━━ 1s 504us/step


c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1005/1005 ━━━━━━━━━━━━━━━━━━━━ 1s 495us/step


In [32]:
#Store the results in the comparison DataFrame
kdd_tf_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
kdd_final_df = kdd_tf_df.set_index('Metric').T
kdd_final_df.index = ['Keras-Tensorflow'] 
print(kdd_final_df)

Metric                Accuracy Balanced Accuracy        Recall     Precision  \
Keras-Tensorflow  86.01 ± 0.09      72.62 ± 0.62  86.01 ± 0.09  85.10 ± 0.07   

Metric                     AUC      F1 Score  
Keras-Tensorflow  72.62 ± 0.62  91.54 ± 0.05  


#### KDD (Experiment 2)

In [33]:
#Modify the training and test variables to use the xuetangx dataset for readability
X_train, X_test, y_train, y_test = kdd_expanded_X_train, kdd_expanded_X_test, kdd_expanded_y_train, kdd_expanded_y_test

In [34]:
#Initialize KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

#Prepare results storage
results = {
    'Accuracy': [],
    'Balanced Accuracy': [],
    'Recall': [],
    'Precision': [],
    'AUC': [],
    'F1 Score': []
}

X_np = X_train.values
y_np = y_train.values

for train_index, val_index in kf.split(X_np):
    X_train_fold, X_val_fold = X_np[train_index], X_np[val_index]
    y_train_fold, y_val_fold = y_np[train_index], y_np[val_index]

    #Build and compile model
    model = Sequential([
        Dense(128, kernel_regularizer=l2(0.001), activation='relu', input_shape=(X_train.shape[1],)),
        BatchNormalization(),
        Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dense(len(np.unique(y_train)), activation='softmax')
    ])
    
    model.compile(optimizer='adam', loss='categorical_crossentropy')

    #Convert y to one-hot for training
    y_train_cat = to_categorical(y_train_fold)
    
    #Fit the model
    model.fit(X_train_fold, y_train_cat, epochs=100, verbose=0, batch_size=512)

    #Predict and evaluate
    y_val_probs = model.predict(X_val_fold)
    y_pred = np.argmax(y_val_probs, axis=1)

    #Metrics
    acc = accuracy_score(y_val_fold, y_pred) * 100
    bal_acc = balanced_accuracy_score(y_val_fold, y_pred) * 100
    rec = recall_score(y_val_fold, y_pred, average='weighted') * 100
    prec = precision_score(y_val_fold, y_pred, average='weighted') * 100
    auc = roc_auc_score(y_val_fold, y_pred) * 100
    f1 = f1_score(y_val_fold, y_pred) * 100

    #Store metrics
    results['Accuracy'].append(acc)
    results['Balanced Accuracy'].append(bal_acc)
    results['Recall'].append(rec)
    results['Precision'].append(prec)
    results['AUC'].append(auc)
    results['F1 Score'].append(f1)

#Summarize results
summary = {
    metric: f"{np.mean(vals):.2f} ± {np.std(vals):.2f}"
    for metric, vals in results.items()
}



c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


603/603 ━━━━━━━━━━━━━━━━━━━━ 0s 534us/step


c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


603/603 ━━━━━━━━━━━━━━━━━━━━ 0s 538us/step


c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


603/603 ━━━━━━━━━━━━━━━━━━━━ 0s 514us/step


c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


603/603 ━━━━━━━━━━━━━━━━━━━━ 0s 515us/step


c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


603/603 ━━━━━━━━━━━━━━━━━━━━ 0s 543us/step


In [35]:
#Store the results in the comparison DataFrame
kdd_expanded_tf_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
kdd_expanded_final_df = kdd_expanded_tf_df.set_index('Metric').T
kdd_expanded_final_df.index = ['Keras-Tensorflow'] 
print(kdd_expanded_final_df)

Metric                Accuracy Balanced Accuracy        Recall     Precision  \
Keras-Tensorflow  85.50 ± 0.24      73.12 ± 0.19  85.50 ± 0.24  84.53 ± 0.24   

Metric                     AUC      F1 Score  
Keras-Tensorflow  73.12 ± 0.19  91.15 ± 0.17  


### FastAI

#### Xuetangx

In [36]:
#Modify the training and test variables to use the xuetangx dataset for readability
X_train, X_test, y_train, y_test = xuetangx_X_train, xuetangx_X_test, xuetangx_y_train, xuetangx_y_test

In [37]:
#Create a DNN model using FastAI
#Initialize results dictionary first
results = {
    'Accuracy': [],
    'Balanced Accuracy': [],
    'Recall': [],
    'Precision': [],
    'AUC': [],
    'F1 Score': []
}

splits = RandomSplitter(valid_pct=0.2)(range_of(X_train))
X = X_train.copy()
X['truth'] = y_train.values
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in kf.split(X):
    train_df = X.iloc[train_idx].copy()
    val_df = X.iloc[val_idx].copy()

    #FastAI Tabular setup
    splits = (list(range(len(train_df))), list(range(len(train_df), len(train_df) + len(val_df))))
    full_df = pd.concat([train_df, val_df])
    
    tp = TabularPandas(
        full_df,
        procs=[],
        cont_names=list(X_train.columns),
        y_names='truth',
        splits=splits
    )

    dls = tp.dataloaders(bs=64)

    learn = tabular_learner(dls, metrics=accuracy)
    learn.fit_one_cycle(5)
    
    # Predictions
    X_val = val_df.drop(columns='truth')
    y_true = val_df['truth'].values
    dl_test = learn.dls.test_dl(X_val)
    
    preds = learn.get_preds(dl=dl_test)[0]
    
    y_pred = (preds >= 0.5).int().numpy()
    
    #Metrics
    results['Accuracy'].append(accuracy_score(y_true, y_pred) * 100)
    results['Balanced Accuracy'].append(balanced_accuracy_score(y_true, y_pred) * 100)
    results['Recall'].append(recall_score(y_true, y_pred, average='weighted') * 100)
    results['Precision'].append(precision_score(y_true, y_pred, average='weighted') * 100)
    results['AUC'].append(roc_auc_score(y_true, y_pred) * 100)
    results['F1 Score'].append(f1_score(y_true, y_pred) * 100)
    results['F1 Score'].append(f1_score(y_true, y_pred) * 100)

#Summarize results
summary = {
    metric: f"{np.mean(vals):.2f} ± {np.std(vals):.2f}"
    for metric, vals in results.items()
}



epoch,train_loss,valid_loss,accuracy,time
0,0.137930,0.143591,0.239758,00:09
1,0.127420,0.142143,0.239758,00:09
2,0.126596,0.133010,0.239758,00:09
3,0.125720,0.128985,0.239758,00:09
4,0.125327,0.371244,0.239758,00:09


epoch,train_loss,valid_loss,accuracy,time
0,0.135603,0.259879,0.241393,00:09
1,0.130498,0.335425,0.241393,00:09
2,0.120456,0.461244,0.241393,00:09
3,0.123469,0.138042,0.241393,00:09
4,0.125880,1.017429,0.241393,00:09


epoch,train_loss,valid_loss,accuracy,time
0,0.136419,0.137663,0.243886,00:09
1,0.128579,0.134822,0.243886,00:09
2,0.123811,0.136981,0.243886,00:09
3,0.124800,0.155129,0.243886,00:09
4,0.118331,0.163183,0.243886,00:09


epoch,train_loss,valid_loss,accuracy,time
0,0.137159,0.141835,0.240873,00:09
1,0.128424,0.130354,0.240873,00:09
2,0.119774,0.155484,0.240873,00:09
3,0.119300,0.128185,0.240873,00:09
4,0.124849,0.130494,0.240873,00:09


epoch,train_loss,valid_loss,accuracy,time
0,0.130357,0.177905,0.244419,00:09
1,0.126359,0.141551,0.244419,00:09
2,0.124625,0.130979,0.244419,00:09
3,0.120798,0.130865,0.244419,00:09
4,0.125301,0.154158,0.244419,00:09


In [38]:
#Store the results in the comparison DataFrame
xuetangx_fai_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
xuetangx_fai_df = xuetangx_fai_df.set_index('Metric').T
xuetangx_fai_df.index = ['Fast.AI']
#Append the results to the existing DataFrame
xuetangx_final_df = pd.concat([xuetangx_final_df, xuetangx_fai_df], axis=0)
print(xuetangx_final_df)

Metric                Accuracy Balanced Accuracy        Recall     Precision  \
Keras-Tensorflow  83.71 ± 0.10      71.84 ± 0.21  83.71 ± 0.10  82.86 ± 0.14   
Fast.AI           83.36 ± 0.16      69.77 ± 0.81  83.36 ± 0.16  82.73 ± 0.12   

Metric                     AUC      F1 Score  
Keras-Tensorflow  71.84 ± 0.21  89.82 ± 0.09  
Fast.AI           69.77 ± 0.81  89.75 ± 0.07  


#### KDD (Experiment 1)

In [39]:
#Modify the training and test variables to use the kdd dataset for readability
X_train, X_test, y_train, y_test = kdd_X_train, kdd_X_test, kdd_y_train, kdd_y_test

In [40]:
#Create a DNN model using FastAI
#Initialize results dictionary first
results = {
    'Accuracy': [],
    'Balanced Accuracy': [],
    'Recall': [],
    'Precision': [],
    'AUC': [],
    'F1 Score': []
}

splits = RandomSplitter(valid_pct=0.2)(range_of(X_train))
X = X_train.copy()
X['truth'] = y_train.values
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in kf.split(X):
    train_df = X.iloc[train_idx].copy()
    val_df = X.iloc[val_idx].copy()

    #FastAI Tabular setup
    splits = (list(range(len(train_df))), list(range(len(train_df), len(train_df) + len(val_df))))
    full_df = pd.concat([train_df, val_df])
    
    tp = TabularPandas(
        full_df,
        procs=[],
        cont_names=list(X_train.columns),
        y_names='truth',
        splits=splits
    )

    dls = tp.dataloaders(bs=64)

    learn = tabular_learner(dls, metrics=accuracy)
    learn.fit_one_cycle(5)
    
    #Predictions
    X_val = val_df.drop(columns='truth')
    y_true = val_df['truth'].values
    dl_test = learn.dls.test_dl(X_val)
    
    preds = learn.get_preds(dl=dl_test)[0]
    
    #For binary classification
    y_pred = (preds >= 0.5).int().numpy()
    
    #Metrics
    results['Accuracy'].append(accuracy_score(y_true, y_pred) * 100)
    results['Balanced Accuracy'].append(balanced_accuracy_score(y_true, y_pred) * 100)
    results['Recall'].append(recall_score(y_true, y_pred, average='weighted') * 100)
    results['Precision'].append(precision_score(y_true, y_pred, average='weighted') * 100)
    results['AUC'].append(roc_auc_score(y_true, y_pred) * 100)
    results['F1 Score'].append(f1_score(y_true, y_pred) * 100)
    results['F1 Score'].append(f1_score(y_true, y_pred) * 100)

#Summarize results
summary = {
    metric: f"{np.mean(vals):.2f} ± {np.std(vals):.2f}"
    for metric, vals in results.items()
}

epoch,train_loss,valid_loss,accuracy,time
0,0.115492,0.123064,0.208275,00:08
1,0.116153,0.111537,0.208275,00:08
2,0.111262,0.107836,0.208275,00:08
3,0.105925,0.106859,0.208275,00:08
4,0.107100,0.106309,0.208275,00:08


epoch,train_loss,valid_loss,accuracy,time
0,0.111122,0.120863,0.209208,00:08
1,0.110811,0.119589,0.209208,00:08
2,0.116710,0.109772,0.209208,00:08
3,0.109502,0.108577,0.209208,00:08
4,0.110770,0.108076,0.209208,00:08


epoch,train_loss,valid_loss,accuracy,time
0,0.120382,0.128549,0.202022,00:08
1,0.114238,0.108749,0.202022,00:08
2,0.113380,0.108497,0.202022,00:08
3,0.109530,0.107032,0.202022,00:08
4,0.108607,0.107273,0.202022,00:08


epoch,train_loss,valid_loss,accuracy,time
0,0.121817,0.127003,0.210833,00:08
1,0.108052,0.108994,0.210833,00:08
2,0.108901,0.107331,0.210833,00:08
3,0.110905,0.107941,0.210833,00:08
4,0.107907,0.106763,0.210833,00:08


epoch,train_loss,valid_loss,accuracy,time
0,0.118457,0.145522,0.206944,00:08
1,0.110339,0.109376,0.206944,00:08
2,0.111777,0.108114,0.206944,00:08
3,0.108116,0.108567,0.206944,00:08
4,0.115869,0.107928,0.206944,00:08


In [41]:
#Store the results in the comparison DataFrame
kdd_fai_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
kdd_fai_df = kdd_fai_df.set_index('Metric').T
kdd_fai_df.index = ['Fast.AI']
#Append the results to the existing DataFrame
kdd_final_df = pd.concat([kdd_final_df, kdd_fai_df], axis=0)
print(kdd_final_df)

Metric                Accuracy Balanced Accuracy        Recall     Precision  \
Keras-Tensorflow  86.01 ± 0.09      72.62 ± 0.62  86.01 ± 0.09  85.10 ± 0.07   
Fast.AI           85.93 ± 0.13      72.28 ± 0.43  85.93 ± 0.13  84.99 ± 0.16   

Metric                     AUC      F1 Score  
Keras-Tensorflow  72.62 ± 0.62  91.54 ± 0.05  
Fast.AI           72.28 ± 0.43  91.50 ± 0.12  


#### KDD (Experiment 2)

In [42]:
#Modify the training and test variables to use the kdd_expanded dataset for readability
X_train, X_test, y_train, y_test = kdd_expanded_X_train, kdd_expanded_X_test, kdd_expanded_y_train, kdd_expanded_y_test

In [43]:
#Create a DNN model using FastAI
#Initialize results dictionary first
results = {
    'Accuracy': [],
    'Balanced Accuracy': [],
    'Recall': [],
    'Precision': [],
    'AUC': [],
    'F1 Score': []
}

splits = RandomSplitter(valid_pct=0.2)(range_of(X_train))
X = X_train.copy()
X['truth'] = y_train.values
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in kf.split(X):
    train_df = X.iloc[train_idx].copy()
    val_df = X.iloc[val_idx].copy()

    #FastAI Tabular setup
    splits = (list(range(len(train_df))), list(range(len(train_df), len(train_df) + len(val_df))))
    full_df = pd.concat([train_df, val_df])
    
    tp = TabularPandas(
        full_df,
        procs=[],
        cat_names=[],  # If you have categorical features, add them here
        cont_names=list(X_train.columns),
        y_names='truth',
        splits=splits
    )

    dls = tp.dataloaders(bs=64)

    learn = tabular_learner(dls, metrics=accuracy)
    learn.fit_one_cycle(5)
    
    # Predictions
    X_val = val_df.drop(columns='truth')
    y_true = val_df['truth'].values
    dl_test = learn.dls.test_dl(X_val)
    
    # Fix: get_preds returns a tuple, extract just the predictions
    preds = learn.get_preds(dl=dl_test)[0]
    
    # For binary classification
    y_pred = (preds >= 0.5).int().numpy()
    
    #Metrics
    results['Accuracy'].append(accuracy_score(y_true, y_pred) * 100)
    results['Balanced Accuracy'].append(balanced_accuracy_score(y_true, y_pred) * 100)
    results['Recall'].append(recall_score(y_true, y_pred, average='weighted') * 100)
    results['Precision'].append(precision_score(y_true, y_pred, average='weighted') * 100)
    results['AUC'].append(roc_auc_score(y_true, y_pred) * 100)
    results['F1 Score'].append(f1_score(y_true, y_pred) * 100)
    results['F1 Score'].append(f1_score(y_true, y_pred) * 100)

#Summarize results
summary = {
    metric: f"{np.mean(vals):.2f} ± {np.std(vals):.2f}"
    for metric, vals in results.items()
}

epoch,train_loss,valid_loss,accuracy,time
0,0.107400,0.117721,0.210919,00:05
1,0.099967,0.102663,0.210919,00:05
2,0.103530,0.109670,0.210919,00:05
3,0.095650,34.930458,0.210919,00:05
4,0.092601,49.145576,0.210919,00:05


epoch,train_loss,valid_loss,accuracy,time
0,0.106883,77.865181,0.204853,00:05
1,0.098463,38.072479,0.204853,00:05
2,0.096521,41.068100,0.204853,00:05
3,0.095403,112.707031,0.204853,00:05
4,0.090202,114.392830,0.204853,00:05


epoch,train_loss,valid_loss,accuracy,time
0,0.111899,121.835121,0.206564,00:05
1,0.101206,119.473885,0.206564,00:05
2,0.094618,87.986496,0.206564,00:05
3,0.092179,100.656151,0.206564,00:05
4,0.093157,148.556458,0.206564,00:05


epoch,train_loss,valid_loss,accuracy,time
0,0.106882,40.128353,0.202375,00:05
1,0.103230,37.807888,0.202375,00:05
2,0.093875,32.061947,0.202375,00:05
3,0.094956,80.121819,0.202375,00:05
4,0.096713,95.147728,0.202375,00:05


epoch,train_loss,valid_loss,accuracy,time
0,0.108711,0.126571,0.210204,00:05
1,0.100598,0.898653,0.210204,00:05
2,0.101796,0.116851,0.210204,00:05
3,0.094305,0.405861,0.210204,00:05
4,0.092389,0.116608,0.210204,00:05


In [44]:
#Store the results in the comparison DataFrame
kdd_expanded_fai_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
kdd_expanded_fai_df = kdd_expanded_fai_df.set_index('Metric').T
kdd_expanded_fai_df.index = ['Fast.AI']
#Append the results to the existing DataFrame
kdd_expanded_final_df = pd.concat([kdd_expanded_final_df, kdd_expanded_fai_df], axis=0)
print(kdd_expanded_final_df)

Metric                Accuracy Balanced Accuracy        Recall     Precision  \
Keras-Tensorflow  85.50 ± 0.24      73.12 ± 0.19  85.50 ± 0.24  84.53 ± 0.24   
Fast.AI           87.36 ± 0.18      75.03 ± 0.45  87.36 ± 0.18  86.67 ± 0.20   

Metric                     AUC      F1 Score  
Keras-Tensorflow  73.12 ± 0.19  91.15 ± 0.17  
Fast.AI           75.03 ± 0.45  92.34 ± 0.13  


In [45]:
#Display the final comparison DataFrame for all three datasets
display(xuetangx_final_df)
display(kdd_final_df)
display(kdd_expanded_final_df)

Metric,Accuracy,Balanced Accuracy,Recall,Precision,AUC,F1 Score
Keras-Tensorflow,83.71 ± 0.10,71.84 ± 0.21,83.71 ± 0.10,82.86 ± 0.14,71.84 ± 0.21,89.82 ± 0.09
Fast.AI,83.36 ± 0.16,69.77 ± 0.81,83.36 ± 0.16,82.73 ± 0.12,69.77 ± 0.81,89.75 ± 0.07


Metric,Accuracy,Balanced Accuracy,Recall,Precision,AUC,F1 Score
Keras-Tensorflow,86.01 ± 0.09,72.62 ± 0.62,86.01 ± 0.09,85.10 ± 0.07,72.62 ± 0.62,91.54 ± 0.05
Fast.AI,85.93 ± 0.13,72.28 ± 0.43,85.93 ± 0.13,84.99 ± 0.16,72.28 ± 0.43,91.50 ± 0.12


Metric,Accuracy,Balanced Accuracy,Recall,Precision,AUC,F1 Score
Keras-Tensorflow,85.50 ± 0.24,73.12 ± 0.19,85.50 ± 0.24,84.53 ± 0.24,73.12 ± 0.19,91.15 ± 0.17
Fast.AI,87.36 ± 0.18,75.03 ± 0.45,87.36 ± 0.18,86.67 ± 0.20,75.03 ± 0.45,92.34 ± 0.13


In [46]:
folder_path = '../../Analysis/Deep_Learning_Results'
xuetangx_final_df.to_csv(os.path.join(folder_path, 'xuetangx_final_results.csv'))
kdd_final_df.to_csv(os.path.join(folder_path, 'kdd_final_results.csv'))
kdd_expanded_final_df.to_csv(os.path.join(folder_path, 'kdd_expanded_final_results.csv'))

## SVM

Implemented for team members due to having more computational resources. Results are saved as csv files in the `Analysis` folder for members to append to their own results.

### Xuetangx SVM

In [47]:
#Modify the training and test variables to use the xuetangx dataset for readability
X_train, X_test, y_train, y_test = xuetangx_X_train, xuetangx_X_test, xuetangx_y_train, xuetangx_y_test

In [48]:
#Define the model
svc = SVC(gamma='auto')
#Define the KFold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
#Perform cross-validation
scores = cross_val_score(svc, X_train, y_train, cv=kf, scoring='accuracy', n_jobs=-1)
#Calculate the mean and standard deviation of the scores
mean_acc = np.mean(scores)
std_acc = np.std(scores)

In [49]:
#Store the results into a dataframe
xuetangx_svm_df = pd.DataFrame([{
    'Classifiers': 'SVM',
    'Mean Accuracy': round(mean_acc, 5),
    'SD': round(std_acc, 5)
}])

In [50]:
#Save the results to a new CSV file
folder_path = '../../Analysis/SVM_Results'
xuetangx_svm_df.to_csv(os.path.join(folder_path, 'xuetangx_svm_results.csv'))

#### KDD (Experiment 1) SVM

In [51]:
#Modify the training and test variables to use the kdd dataset for readability
X_train, X_test, y_train, y_test = kdd_X_train, kdd_X_test, kdd_y_train, kdd_y_test

In [52]:
#Define the model
svc = SVC(gamma='auto')
#Define the KFold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
#Perform cross-validation
scores = cross_val_score(svc, X_train, y_train, cv=kf, scoring='accuracy', n_jobs=-1)
#Calculate the mean and standard deviation of the scores
mean_acc = np.mean(scores)
std_acc = np.std(scores)

In [53]:
#Store the results into a dataframe
kdd_svm_df = pd.DataFrame([{
    'Classifiers': 'SVM',
    'Mean Accuracy': round(mean_acc, 5),
    'SD': round(std_acc, 5)
}])

In [54]:
#Save the results to a new CSV file
folder_path = '../../Analysis/SVM_Results'
kdd_svm_df.to_csv(os.path.join(folder_path, 'kdd_svm_results.csv'))

### KDD (Experiment 2) SVM

In [55]:
#Modify the training and test variables to use the kdd_expanded dataset for readability
X_train, X_test, y_train, y_test = kdd_expanded_X_train, kdd_expanded_X_test, kdd_expanded_y_train, kdd_expanded_y_test

In [56]:
#Define the model
svc = SVC(gamma='auto')
#Define the KFold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
#Perform cross-validation
scores = cross_val_score(svc, X_train, y_train, cv=kf, scoring='accuracy', n_jobs=-1)
#Calculate the mean and standard deviation of the scores
mean_acc = np.mean(scores)
std_acc = np.std(scores)

In [57]:
#Store the results into a dataframe
kdd_expanded_svm_df = pd.DataFrame([{
    'Classifiers': 'SVM',
    'Mean Accuracy': round(mean_acc, 5),
    'SD': round(std_acc, 5)
}])

In [58]:
#Save the results to a new CSV file
folder_path = '../../Analysis/SVM_Results'
kdd_expanded_svm_df.to_csv(os.path.join(folder_path, 'kdd_expanded_svm_results.csv'))